### Invoice Genie Space

This stage deploys a Databricks Genie space connected to the processed procurement
Delta tables. Users can ask natural language questions about supplier spend, invoice
status, payment aging, billing discrepancies, discount capture rates, and SLA performance.

In [ ]:
%pip install --upgrade databricks-sdk

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

##### Create a SQL warehouse for the Genie space

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

import sys
sys.path.append('../utils')
from uc_state import add

WAREHOUSE_NAME = f"{CATALOG}-procurement-warehouse"

existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if existing_wh:
    warehouse = existing_wh[0]
    print(f"Using existing warehouse: {warehouse.id}")
else:
    warehouse = w.warehouses.create(
        name=WAREHOUSE_NAME,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True,
    ).result()
    print(f"\u2705 Created warehouse: {warehouse.id}")

add(CATALOG, "warehouses", warehouse)

##### Deploy the Genie space via REST API

In [ ]:
import json
import hashlib

GENIE_SPACE_TITLE = f"Caspers Procurement Intelligence ({CATALOG})"

GENIE_DESCRIPTION = """You are an accounts payable analyst for Caspers Kitchens, a ghost kitchen network
operating 16 restaurant brands across 4 locations. You have access to procurement data:
supplier invoices, contract price schedules, and financial exception reports organized
in a bronze/silver/gold medallion architecture.

=== DATA MODEL ===

SOURCE TABLES (raw, pre-pipeline):

suppliers: One row per vendor.
  Key columns: supplier_id, name, category (Fresh Produce / Meat & Poultry / Poultry /
  Seafood / Dairy & Eggs / Packaging / Beverages / Cleaning & Sanitation /
  Dry Goods & Pantry / Spices & Seasonings), payment_terms, contract_id.

contract_price_schedule: Contracted unit prices per item per vendor.
  Key columns: contract_id, supplier_id, supplier_name, category, item, unit,
  contract_unit_price.

SILVER TABLES (enriched):

silver_invoices: Every invoice with exception flags and financial amounts.
  Key columns: invoice_id, supplier_id, invoice_date, due_date, payment_date, status
  (paid / outstanding / past_due / disputed), flags (array), attention_required (boolean),
  is_paid, is_outstanding, is_overdue, is_disputed, is_open,
  has_missed_discount, has_missing_volume_discount, has_price_discrepancy,
  has_late_fee, has_sla_penalty,
  subtotal, discount_amount, sla_penalty, late_fee, total_due,
  missed_discount, volume_discount_owed, price_discrepancy,
  recoverable_amount (missed_discount + volume_discount_owed + price_discrepancy),
  days_since_invoice, days_overdue.

silver_line_items: One row per line item per invoice.
  Key columns: invoice_id, supplier_id, description, qty, unit, unit_price,
  contract_price, line_total, contract_line_total,
  price_variance (line_total - contract_line_total),
  discrepancy_pct (% over/under contract price),
  variance_direction (overbilled / underbilled / correct),
  is_discrepant (boolean).

GOLD TABLES (business-ready):

invoices: Invoice summary joined with supplier name and category.
  Use for: invoice-level queries, status filtering, amount lookups.

line_items: Line item detail joined with supplier and invoice context.
  Use for: drilling into specific overbilled items, comparing unit prices to contract.

spend_by_supplier: Aggregate spend per vendor.
  Key columns: supplier_name, supplier_category, total_invoices, paid_invoices,
  open_invoices, disputed_invoices, total_billed, total_paid, open_balance,
  total_recoverable, avg_invoice_value.

spend_by_category: Aggregate spend per supplier category.
  Key columns: supplier_category, total_invoices, total_billed, open_balance,
  avg_invoice_value, min_invoice_value, max_invoice_value.

payment_aging: Open invoices only, bucketed by days overdue.
  Key columns: invoice_id, supplier_name, invoice_date, due_date, days_overdue,
  aging_bucket (current / 1-30 days overdue / 31-60 days overdue / 60+ days overdue),
  status, total_due, late_fee.

invoice_exceptions: All invoices with at least one financial exception — the AP review queue.
  Key columns: invoice_id, supplier_name, invoice_date, status,
  has_price_discrepancy, has_missed_discount, has_missing_volume_discount,
  has_late_fee, has_sla_penalty,
  price_discrepancy, missed_discount, volume_discount_owed,
  late_fee, sla_penalty, recoverable_amount, days_overdue.

supplier_scorecard: Per-supplier performance summary.
  Key columns: supplier_name, supplier_category, payment_terms, total_invoices,
  disputed_invoices, overdue_invoices, sla_violations, price_discrepancies,
  total_late_fees_accrued, total_sla_penalties, total_overbilled,
  total_recoverable, discounts_captured, discounts_missed.

discount_analysis: Discount capture rate per supplier.
  Key columns: supplier_name, supplier_payment_terms, discount_eligible_invoices,
  discounts_captured_count, early_pay_discounts_missed_count,
  volume_discounts_missed_count, total_discount_captured,
  total_early_pay_discount_missed, total_volume_discount_missed,
  total_discount_opportunity, discount_capture_rate_pct.

=== QUERY GUIDELINES ===

- For overdue invoices: use payment_aging WHERE aging_bucket != 'current'.
- For billing disputes: use invoice_exceptions WHERE has_price_discrepancy = true
  or line_items WHERE is_discrepant = true.
- For total outstanding liability: SUM(total_due) from silver_invoices WHERE is_open = true.
- For missed discounts: use discount_analysis or invoice_exceptions.
- For supplier comparison: use supplier_scorecard or spend_by_supplier.
- recoverable_amount = price_discrepancy + missed_discount + volume_discount_owed.
- Statuses: paid, outstanding (open, not yet due), past_due (overdue), disputed
  (payment withheld pending credit memo or corrected invoice).
- Supplier IDs: VFP=Valley Farms Produce, PCM=Prime Cut Meats, HPC=Heritage Poultry,
  PCS=Pacific Coast Seafood, GSD=Golden State Dairy, EPK=EcoPack Solutions,
  MSB=Mountain Spring Beverages, PSS=ProSan Supplies, CTF=Continental Foods,
  SRT=Spice Route Trading."""

TABLE_NAMES = sorted([
    f"{CATALOG}.procurement.suppliers",
    f"{CATALOG}.procurement.contract_price_schedule",
    f"{CATALOG}.procurement.silver_invoices",
    f"{CATALOG}.procurement.silver_line_items",
    f"{CATALOG}.procurement.invoices",
    f"{CATALOG}.procurement.line_items",
    f"{CATALOG}.procurement.spend_by_supplier",
    f"{CATALOG}.procurement.spend_by_category",
    f"{CATALOG}.procurement.payment_aging",
    f"{CATALOG}.procurement.invoice_exceptions",
    f"{CATALOG}.procurement.supplier_scorecard",
    f"{CATALOG}.procurement.discount_analysis",
])

serialized_space = json.dumps({
    "version": 2,
    "data_sources": {
        "tables": [{"identifier": t} for t in TABLE_NAMES],
    },
    "config": {
        "sample_questions": [
            {"id": hashlib.md5(q.encode()).hexdigest(), "question": [q]}
            for q in [
                "Show me all invoices that are currently past due or disputed",
                "Which suppliers have billing discrepancies against contracted prices?",
                "What is our total open balance by supplier?",
                "How much money have we missed in early payment and volume discounts?",
                "Which supplier categories represent our highest total spend?",
                "Show me all invoice exceptions ranked by recoverable amount",
                "What is the discount capture rate for each eligible supplier?",
                "Which invoices have SLA penalties applied, and how much was deducted?",
                "Show me the payment aging breakdown — how much is 30+ days overdue?",
                "Which supplier has the most billing discrepancies?",
            ]
        ],
    },
})

genie_space_id = None
try:
    state_df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'genie_spaces'
        ORDER BY created_at DESC LIMIT 1
    """)
    if state_df.count() > 0:
        genie_info = json.loads(state_df.first().resource_data)
        candidate_id = genie_info.get("space_id")
        if candidate_id:
            try:
                w.genie.get_space(candidate_id)
                genie_space_id = candidate_id
                print(f"Found existing Genie space: {genie_space_id}")
            except Exception:
                print(f"Genie space {candidate_id} in state but no longer exists — will recreate")
except Exception:
    pass

if not genie_space_id:
    space = w.genie.create_space(
        warehouse_id=warehouse.id,
        serialized_space=serialized_space,
        title=GENIE_SPACE_TITLE,
        description=GENIE_DESCRIPTION,
    )
    genie_space_id = space.space_id
    print(f"\u2705 Created Genie space: {genie_space_id}")

print(f"   Title: {GENIE_SPACE_TITLE}")
print(f"   Tables: {len(TABLE_NAMES)} tables in {CATALOG}.procurement")

add(CATALOG, "genie_spaces", {"space_id": genie_space_id, "title": GENIE_SPACE_TITLE})
print("\u2705 Invoice Genie stage complete")